In [1]:
import torch 
import sys
from datasets.graph_datasets.graph_heat_dataset import HeatGraphDataset
import yaml
from models.forecasting.GNO import GNO
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data, Batch
from datasets.graph_datasets.graph_data_utils import partition_domain_into_subgraphs, get_subgraph_grid_size, run_total_domain_inference
sys.path.append('...')

/Users/louisgodtfredsen/Desktop/Coding Projects/ML-for-PDEs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load configs
with open("../checkpoints/gno_heat/20260912_2148/model_configs.yaml", "r") as file:
    cfg = yaml.safe_load(file)

# Load in sample to run inference on

In [3]:
data_path ='../data/test_data/heat_equation_m64_h0_minmax_N200.pt'
input_data = torch.load(data_path)
simulation_idx = 0
simulation_frame = 0
field_keys = list(input_data.keys())[1:]

X, X_t1 = input_data['X'][simulation_idx,simulation_frame], input_data['X'][simulation_idx,simulation_frame+1]
pde_params = [float(input_data[k][simulation_idx]) for k in field_keys]

H, W = X.shape[-1], X.shape[-2] 
x_indices = torch.tensor([x for x in range(W)])
y_indices = torch.tensor([x for x in range(H)])
node_grid_indices = torch.cartesian_prod(x_indices, y_indices) # Collection of (x, y) grid indices
node_spatial_pos = torch.cartesian_prod(x_indices / W, y_indices / H) # Collection of (x, y) spatial positions

dataset = HeatGraphDataset('../data/test_data/heat_equation_m64_h0_minmax_N200.pt',
                           list(input_data.keys())[1:],
                           r = cfg['radius'],
                           bc ='periodic',
                           sub_graph_size = cfg['sub_graph_size'])

In [4]:
get_subgraph_grid_size(200)

(10, 20)

In [5]:
subgraphs = partition_domain_into_subgraphs(X, X_t1, H, W, num_subgraph_nodes=200, r = 0.025, boundary_condition='periodic', pde_params=pde_params)

In [6]:
subgraphs

[Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200),
 Data(x=[200, 3], edg

# Load Model

In [7]:
gno_model = GNO(optimiser = cfg['optimiser'], 
                 learning_rate = cfg['learning_rate'], 
                 num_node_input_features = cfg['num_node_input_features'],
                 num_edge_features = cfg['num_edge_features'], 
                 num_latent_dim = cfg['num_latent_dim'], 
                 output_dim = cfg['output_dim'],
                 num_gno_layers = cfg['num_gno_layers'],
                 kernel_ffn_layers = cfg['kernel_ffn_layers'],
                 kernel_ffn_dropout = cfg['kernel_ffn_dropout'], 
                 GNO_layer_activation = cfg['gno_layer_activation'])

model_path = '../checkpoints/gno_heat/20260912_2148/gno-epoch=0009-val_loss=0.0000.ckpt'
gno_model.load_state_dict(torch.load(model_path)['state_dict'])

<All keys matched successfully>

In [8]:
y_hat = gno_model(Batch.from_data_list(subgraphs)).reshape(-1, 10, 20)

In [9]:
y_t = gno_model(subgraphs[0])

In [10]:
y_hat[0][1, :]

tensor([1.0845e-01, 8.0451e-02, 5.7780e-02, 3.9653e-02, 2.6071e-02, 1.6519e-02,
        1.0117e-02, 5.9876e-03, 3.4383e-03, 1.9352e-03, 1.0865e-03, 6.2573e-04,
        3.8478e-04, 2.6292e-04, 2.0275e-04, 1.7333e-04, 1.5725e-04, 1.4535e-04,
        1.2051e-04, 9.2685e-05], grad_fn=<SelectBackward0>)

In [11]:
total_domain_output = run_total_domain_inference(Batch.from_data_list(subgraphs), torch.zeros((64,64)), gno_model)

Output shape: torch.Size([28, 10, 20])
X_idx: 0, 10
y_idx: 0, 20
X_idx: 0, 10
y_idx: 20, 40
X_idx: 0, 10
y_idx: 40, 60
X_idx: 10, 20
y_idx: 0, 20
X_idx: 10, 20
y_idx: 20, 40
X_idx: 20, 30
y_idx: 0, 20


In [ ]:
total_domain_output[0, :20]

torch.Size([20])

In [41]:
gno_model(subgraphs[4])[0:20] == total_domain_output[10, 0:20].unsqueeze(-1)

tensor([[True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True]])